# Taller de código 1: algoritmos de búsqueda

Implementarás **BFS, UCS y A\*** sobre el mismo grafo. El objetivo no es solo encontrar una ruta, sino comparar:

- ruta encontrada;
- costo total;
- número y orden de nodos expandidos.

Tiempo estimado: **60–75 minutos**.

## Problema

Un agente debe viajar desde `S` hasta `G`. Las aristas tienen costos y cada nodo posee una heurística estimada hacia la meta.

| Nodo | $h(n)$ |
|---|---:|
| S | 6 |
| A | 4 |
| B | 3 |
| C | 5 |
| D | 2 |
| G | 0 |

In [ ]:
from collections import deque
import heapq
import itertools
import pandas as pd
import matplotlib.pyplot as plt

graph = {
    "S": [("A", 2), ("B", 5)],
    "A": [("C", 2), ("D", 4)],
    "B": [("D", 1)],
    "C": [("G", 5)],
    "D": [("G", 2)],
    "G": [],
}

heuristic = {"S": 6, "A": 4, "B": 3, "C": 5, "D": 2, "G": 0}

In [ ]:
positions = {"S": (0, 1), "A": (1, 2), "B": (1, 0), "C": (2, 3), "D": (2, 1), "G": (3, 2)}

plt.figure(figsize=(8, 4))
for source, neighbors in graph.items():
    x1, y1 = positions[source]
    for target, cost in neighbors:
        x2, y2 = positions[target]
        plt.plot([x1, x2], [y1, y2], color="#95a5a6", zorder=1)
        plt.text((x1+x2)/2, (y1+y2)/2 + 0.12, str(cost), ha="center")
for node, (x, y) in positions.items():
    plt.scatter(x, y, s=900, color="#2980b9", zorder=2)
    plt.text(x, y, node, color="white", ha="center", va="center", fontsize=13, weight="bold")
plt.axis("off")
plt.title("Grafo de búsqueda: números sobre las aristas = costos")
plt.show()

## Actividad 1 — Reconstruir una ruta

In [ ]:
def reconstruct_path(parent, goal):
    """Reconstruye la ruta desde el inicio hasta goal usando parent."""
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = parent.get(curr)
    return path[::-1]

In [ ]:
# Prueba
example_parent = {"S": None, "A": "S", "D": "A", "G": "D"}
assert reconstruct_path(example_parent, "G") == ["S", "A", "D", "G"]
print("Prueba 1 superada.")

## Actividad 2 — BFS

Completa BFS usando una cola FIFO. Ignora los costos al decidir qué nodo expandir, pero calcula el costo de la ruta encontrada al final.

In [ ]:
def path_cost(graph, path):
    total = 0
    for source, target in zip(path, path[1:]):
        total += dict(graph[source])[target]
    return total


def bfs(graph, start, goal):
    """Devuelve path, cost y expanded_order."""
    queue = deque([start])
    parent = {start: None}
    visited = {start}
    expanded_order = []

    while queue:
        curr = queue.popleft()
        expanded_order.append(curr)
        if curr == goal:
            break
        for neighbor, _ in graph.get(curr, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = curr
                queue.append(neighbor)

    path = reconstruct_path(parent, goal)
    cost = path_cost(graph, path)
    return path, cost, expanded_order

In [ ]:
path_bfs, cost_bfs, expanded_bfs = bfs(graph, "S", "G")
assert path_bfs[0] == "S" and path_bfs[-1] == "G"
assert len(path_bfs) == 4
print(path_bfs, cost_bfs, expanded_bfs)

## Actividad 3 — Uniform Cost Search

Usa una cola de prioridad ordenada por $g(n)$. Si encuentras una ruta más barata hacia un nodo, actualiza su mejor costo y su padre.

In [ ]:
def uniform_cost_search(graph, start, goal):
    """Devuelve path, cost y expanded_order."""
    counter = itertools.count()
    # (costo_g, contador_desempate, nodo, padre)
    frontier = [(0, next(counter), start, None)]
    best_cost = {start: 0}
    parent = {}
    expanded_order = []

    while frontier:
        g, _, curr, p = heapq.heappop(frontier)
        if curr in expanded_order:
            continue
        expanded_order.append(curr)
        parent[curr] = p

        if curr == goal:
            break

        for neighbor, step_cost in graph.get(curr, []):
            new_g = g + step_cost
            if neighbor not in best_cost or new_g < best_cost[neighbor]:
                best_cost[neighbor] = new_g
                heapq.heappush(frontier, (new_g, next(counter), neighbor, curr))

    path = reconstruct_path(parent, goal)
    cost = best_cost[goal]
    return path, cost, expanded_order

In [ ]:
path_ucs, cost_ucs, expanded_ucs = uniform_cost_search(graph, "S", "G")
assert cost_ucs == 8
assert path_ucs[0] == "S" and path_ucs[-1] == "G"
print(path_ucs, cost_ucs, expanded_ucs)

## Actividad 4 — A*

Implementa A* usando $f(n)=g(n)+h(n)$. La heurística se usa para ordenar la frontera, pero el costo acumulado real sigue siendo $g(n)$.

In [ ]:
def a_star(graph, heuristic, start, goal):
    """Devuelve path, cost y expanded_order."""
    counter = itertools.count()
    # (f_cost, contador_desempate, g_cost, nodo, padre)
    frontier = [(heuristic[start], next(counter), 0, start, None)]
    best_g = {start: 0}
    parent = {}
    expanded_order = []

    while frontier:
        f, _, g, curr, p = heapq.heappop(frontier)
        if curr in expanded_order:
            continue
        expanded_order.append(curr)
        parent[curr] = p

        if curr == goal:
            break

        for neighbor, step_cost in graph.get(curr, []):
            new_g = g + step_cost
            if neighbor not in best_g or new_g < best_g[neighbor]:
                best_g[neighbor] = new_g
                f_cost = new_g + heuristic[neighbor]
                heapq.heappush(frontier, (f_cost, next(counter), new_g, neighbor, curr))

    path = reconstruct_path(parent, goal)
    cost = best_g[goal]
    return path, cost, expanded_order

In [ ]:
path_astar, cost_astar, expanded_astar = a_star(graph, heuristic, "S", "G")
assert cost_astar == 8
assert path_astar[0] == "S" and path_astar[-1] == "G"
print(path_astar, cost_astar, expanded_astar)

## Actividad 5 — Comparación

In [ ]:
results = pd.DataFrame([
    {"algorithm": "BFS", "path": path_bfs, "cost": cost_bfs, "expanded": len(expanded_bfs)},
    {"algorithm": "UCS", "path": path_ucs, "cost": cost_ucs, "expanded": len(expanded_ucs)},
    {"algorithm": "A*", "path": path_astar, "cost": cost_astar, "expanded": len(expanded_astar)},
])
results

## Preguntas de análisis y respuestas

1. **¿Por qué BFS puede devolver una ruta con mayor costo aunque tenga el mismo número de aristas?**
   - BFS minimiza el número de aristas (pasos/saltos), tratando cada arista como si tuviera peso unitario ($1$). Si dos rutas tienen 3 pasos (por ejemplo `['S', 'A', 'C', 'G']` con costo $2+2+5=9$ y `['S', 'A', 'D', 'G']` con costo $2+4+2=8$), BFS seleccionará la que explore primero en el nivel sin comparar los pesos acumulados reales.

2. **¿Qué diferencia observas entre el orden de expansión de UCS y A\*?**
   - **UCS** expande contornos de igual costo acumulado $g(n)$ de forma radial/uniforme en todas direcciones (`S -> A -> B -> D -> C -> G`).
   - **A\*** utiliza la heurística $h(n)$ para guiar la búsqueda directamente hacia la meta $f(n)=g(n)+h(n)$, enfocando la exploración y reduciendo drásticamente la cantidad de ramas no prometedoras exploradas.

3. **Cambia `heuristic["A"]` a `20`. ¿La heurística sigue siendo admisible? ¿Qué ocurre con el resultado?**
   - El costo real óptimo desde `A` hasta `G` es $4+2=6$ (vía `D`). Como $h(A) = 20 > 6$, la heurística deja de ser admisible (sobreestima el costo real).
   - Como consecuencia, A* penalizará excesivamente expandir `A`, y explorará primero ramas subóptimas como `B -> D` creyendo falsamente que son más prometedoras, perdiendo la garantía formal de optimalidad.

4. **¿En qué parte del código se implementa realmente la estrategia de búsqueda: al generar sucesores o al extraer de la frontera?**
   - Se implementa al **extraer de la frontera**. La regla con la que se prioriza y se retira el siguiente elemento (FIFO en BFS, $\min g(n)$ en UCS, $\min g(n)+h(n)$ en A*) define el comportamiento y la filosofía de búsqueda de cada algoritmo.

## Checklist

- [x] Reconstruí rutas correctamente.
- [x] Implementé BFS con FIFO.
- [x] Implementé UCS con prioridad $g(n)$.
- [x] Implementé A* con prioridad $g(n)+h(n)$.
- [x] Comparé costo y nodos expandidos.